# Random Forest

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score


features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']

train = pd.read_csv('data/splits/train.csv')
val = pd.read_csv('data/splits/val.csv')
test = pd.read_csv('data/splits/test.csv')

X_train, y_train = train[features], train['genre_group']
X_val, y_val = val[features], val['genre_group']
X_test, y_test = test[features], test['genre_group']

print(len(train), len(val), len(test))  # should be roughly 80 / 10 / 10

34316 34317 17159


### using balanced_subsample for class weight

In [4]:
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced_subsample', random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.59      0.69      0.64      4152
             country       0.41      0.56      0.48      1170
    electronic_dance       0.64      0.69      0.66      7728
          hiphop_rnb       0.40      0.28      0.33      1541
          indie_folk       0.33      0.26      0.29      1000
               latin       0.47      0.60      0.53      3671
                 pop       0.42      0.37      0.39      4387
              reggae       0.26      0.26      0.26       776
          rock_metal       0.61      0.57      0.59      6737
               world       0.35      0.23      0.28      3155

            accuracy                           0.53     34317
           macro avg       0.45      0.45      0.44     34317
        weighted avg       0.52      0.53      0.52     34317



### using unbalanced for class weight

In [6]:
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight=None, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.58      0.70      0.63      4152
             country       0.59      0.41      0.48      1170
    electronic_dance       0.57      0.76      0.65      7728
          hiphop_rnb       0.73      0.21      0.32      1541
          indie_folk       0.56      0.15      0.23      1000
               latin       0.50      0.58      0.54      3671
                 pop       0.41      0.37      0.39      4387
              reggae       0.54      0.09      0.15       776
          rock_metal       0.55      0.63      0.59      6737
               world       0.39      0.17      0.24      3155

            accuracy                           0.53     34317
           macro avg       0.54      0.41      0.42     34317
        weighted avg       0.53      0.53      0.51     34317



# testing/tweaking parameters

#### changing min_samples_leaf and max_features

Unbalanced ----------

In [ ]:
best_score = 0
best_params = None

for leaf in [1, 3, 5, 10]:
    for mf in ['sqrt', 0.5, 1.0]:
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, max_features=mf, n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        score = f1_score(y_val, rf.predict(X_val), average='macro')
        print(f"min_samples_leaf={leaf:<3} max_features={mf!s:<5} val macro-F1 = {score:.3f}")

        if score > best_score:
            best_score = score
            best_params = {'min_samples_leaf': leaf, 'max_features': mf}

print("\nBest:", best_params, f"-> {best_score:.3f}")

min_samples_leaf=1   max_features=sqrt  val macro-F1 = 0.436
min_samples_leaf=1   max_features=0.5   val macro-F1 = 0.433
min_samples_leaf=1   max_features=1.0   val macro-F1 = 0.433
min_samples_leaf=3   max_features=sqrt  val macro-F1 = 0.423
min_samples_leaf=3   max_features=0.5   val macro-F1 = 0.426
min_samples_leaf=3   max_features=1.0   val macro-F1 = 0.424
min_samples_leaf=5   max_features=sqrt  val macro-F1 = 0.403
min_samples_leaf=5   max_features=0.5   val macro-F1 = 0.408
min_samples_leaf=5   max_features=1.0   val macro-F1 = 0.410
min_samples_leaf=10  max_features=sqrt  val macro-F1 = 0.376
min_samples_leaf=10  max_features=0.5   val macro-F1 = 0.382
min_samples_leaf=10  max_features=1.0   val macro-F1 = 0.383

Best: {'min_samples_leaf': 1, 'max_features': 'sqrt'} -> 0.436


Balanced ----------

In [16]:
best_score = 0
best_params = None

for leaf in [1, 3, 5, 10]:
    for mf in ['sqrt', 0.5, 1.0]:
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, max_features=mf, class_weight='balanced_subsample', n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        score = f1_score(y_val, rf.predict(X_val), average='macro')
        print(f"min_samples_leaf={leaf:<3} max_features={mf!s:<5} val macro-F1 = {score:.3f}")

        if score > best_score:
            best_score = score
            best_params = {'min_samples_leaf': leaf, 'max_features': mf}

print("\nBest:", best_params, f"-> {best_score:.3f}")

min_samples_leaf=1   max_features=sqrt  val macro-F1 = 0.437
min_samples_leaf=1   max_features=0.5   val macro-F1 = 0.437
min_samples_leaf=1   max_features=1.0   val macro-F1 = 0.436
min_samples_leaf=3   max_features=sqrt  val macro-F1 = 0.443
min_samples_leaf=3   max_features=0.5   val macro-F1 = 0.444
min_samples_leaf=3   max_features=1.0   val macro-F1 = 0.443
min_samples_leaf=5   max_features=sqrt  val macro-F1 = 0.434
min_samples_leaf=5   max_features=0.5   val macro-F1 = 0.434
min_samples_leaf=5   max_features=1.0   val macro-F1 = 0.433
min_samples_leaf=10  max_features=sqrt  val macro-F1 = 0.412
min_samples_leaf=10  max_features=0.5   val macro-F1 = 0.412
min_samples_leaf=10  max_features=1.0   val macro-F1 = 0.413

Best: {'min_samples_leaf': 3, 'max_features': 0.5} -> 0.444


In [17]:
best_score = 0
best_est = 0

for est in [100, 200, 300, 400, 500]:
    rf = RandomForestClassifier(n_estimators=est, min_samples_leaf=best_params['min_samples_leaf'], max_features=best_params['max_features'], class_weight='balanced_subsample', n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train)
    score = f1_score(y_val, rf.predict(X_val), average='macro')
    print(f"n_estimators={est:<3} val macro-F1 = {score:.3f}")

    if score > best_score:
        best_score = score
        best_est = est

print("\nBest:", est, f"-> {best_score:.3f}")

n_estimators=100 val macro-F1 = 0.441
n_estimators=200 val macro-F1 = 0.444
n_estimators=300 val macro-F1 = 0.447
n_estimators=400 val macro-F1 = 0.447
n_estimators=500 val macro-F1 = 0.448

Best: 500 -> 0.448


# Results

In [19]:
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, max_features=0.5, class_weight='balanced_subsample',  n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_val)
print(classification_report(y_val, rf_pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.59      0.69      0.64      4152
             country       0.41      0.56      0.48      1170
    electronic_dance       0.64      0.69      0.66      7728
          hiphop_rnb       0.42      0.29      0.34      1541
          indie_folk       0.33      0.26      0.29      1000
               latin       0.48      0.59      0.53      3671
                 pop       0.42      0.38      0.40      4387
              reggae       0.27      0.27      0.27       776
          rock_metal       0.61      0.57      0.59      6737
               world       0.35      0.23      0.28      3155

            accuracy                           0.53     34317
           macro avg       0.45      0.45      0.45     34317
        weighted avg       0.52      0.53      0.52     34317



In [20]:
print(classification_report(y_test, rf.predict(X_test)))

                      precision    recall  f1-score   support

chill_jazz_classical       0.60      0.69      0.64      2076
             country       0.40      0.52      0.45       585
    electronic_dance       0.65      0.71      0.68      3864
          hiphop_rnb       0.38      0.26      0.31       770
          indie_folk       0.35      0.27      0.30       501
               latin       0.47      0.60      0.53      1835
                 pop       0.42      0.37      0.39      2193
              reggae       0.26      0.25      0.26       388
          rock_metal       0.61      0.57      0.59      3369
               world       0.37      0.24      0.29      1578

            accuracy                           0.53     17159
           macro avg       0.45      0.45      0.44     17159
        weighted avg       0.52      0.53      0.52     17159

